# Importing 

Modules imported for the Neural Network:
- __Numpy__: For complex calculations
- __Rasterio__: For TIFF image processing 
- __PyTorch__: Model functionality, and Dataset creation

In [1]:
import os
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Setting up the Dataset
We create a class that inherits from Dataset. When created, this class:
- Loads samples from samples folder on initialization
- When calling __\_\_getiten\_\___:
    - The function takes an index as an argument
    - It loads the sample and label images with the file name in the index
    - Normalizes the sample image using Rasterio
    - Classifies the label image into one of three classes: No vegetation, low amount of vegetation, and high amount of vegetation
    - Converts images into PyTorch tensors.

In [ ]:
class VegetationDataset(Dataset):
    def __init__(self, samples_dir, labels_dir):
        self.samples_dir = samples_dir
        self.labels_dir = labels_dir
        self.files = os.listdir(samples_dir)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]

        label_name = file_name.replace("_img_", "_ndvi_") 
        label_path = os.path.join(self.labels_dir, label_name)

        # Load sample image
        with rasterio.open(os.path.join(self.samples_dir, file_name)) as src:
            img = src.read().astype(np.float32)  # (3, H, W)

        # Normalization of sample image
        img = img / 255.0

        # Load label image
        with rasterio.open(os.path.join(self.labels_dir, label_name)) as src:
            label = src.read(1).astype(np.float32)  # (H, W)

        # Convert label to class
        # Adjust thresholds if needed
        label_class = np.zeros_like(label, dtype=np.int64)
        label_class[label < 120] = 0
        label_class[(label >= 120) & (label < 170)] = 1
        label_class[label >= 170] = 2

        # Flatten pixels
        img = img.reshape(3, -1).T        # (H*W, 3)
        label_class = label_class.flatten()  # (H*W,)


        return torch.tensor(img), torch.tensor(label_class)

# Loading images

Dataset initialization.

In [18]:
samples_dir = "../../samples"
labels_dir = "../../labels"

dataset = VegetationDataset(samples_dir, labels_dir)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Multi-layer Perception Model Class

This class creates an MLP model that inherits from the nn.Module module. On initialization:
- Initializes nn.Module object
- Sets up model layers using Linear() method
    - First layer with 3 input channels (RGB dimensions and layers) and 32 output channels.
    - Processing layers with 32 input layers and 16 output layers.
    - Output layer with 16 input channels and 3 output layers (classes)
    - We apply ReLU to each layer to add non-linearity.
- We also define the flow of data through the network with the forward() method.

In [19]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3) 
        )

    def forward(self, x):
        return self.net(x)

# Model initialization

- model: Initializes MLP object
- criterion: Loss function. Measures how wrong the preditions are.
- optimizer: Updates weights to reduce loss.

In [20]:
model = MLP()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Model Training

The following code includes all steps to complete model training on the entire dataset. What it does:
- Defines the number of epochs (number of times the model will train over the entire dataset)
- Tracks loss through each epoch by using the loss function
- For each image/label pair in the dataset
    - Flatten image batch
    - Pass each image through the model
    - Calculates loss by comparing outputs with labels
    - Clears old gradients
    - loss.backward() determines how each weight contributed to the error
    - optimizer.step() adjusts model pparameters using gradients
    - total_loss is updated.


In [ ]:
epochs = 20

for epoch in range(epochs):
    total_loss = 0

    for imgs, labels in loader:

        # Flatten batch
        imgs = imgs.view(-1, 3)      # (B * H * W, 3)
        labels = labels.view(-1)     # (B * H * W)

        outputs = model(imgs)        # (B * H * W, 3)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: Loss = {total_loss:.4f}")

Epoch 0: Loss = 138.3858
Epoch 1: Loss = 92.3441
Epoch 2: Loss = 89.5719
Epoch 3: Loss = 89.0892
Epoch 4: Loss = 88.5000
Epoch 5: Loss = 88.3222
Epoch 6: Loss = 88.3255
Epoch 7: Loss = 88.0380
Epoch 8: Loss = 88.0689
Epoch 9: Loss = 87.7651
Epoch 10: Loss = 88.3044
Epoch 11: Loss = 87.9854
Epoch 12: Loss = 87.8706
Epoch 13: Loss = 87.7547
Epoch 14: Loss = 87.1377
Epoch 15: Loss = 87.1102
Epoch 16: Loss = 87.2961
Epoch 17: Loss = 86.8740
Epoch 18: Loss = 87.0860
Epoch 19: Loss = 86.7772


# Model Evaluation

We define the evaluate function. This function:
- defines the number of correctly predicted pixels and the total number of pixels evaluated.
- turns off gradient computation (evaluation without learning, faster evaluation that uses less memory)
- iterates through dataset to:
    - flatten images
    - processes image through defined model
    - compares the output with the most probable class
    - update correct and total variables

In [ ]:
def evaluate(model, loader):
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            # Flatten
            imgs = imgs.view(-1, 3)      
            labels = labels.view(-1)     

            outputs = model(imgs)        
            preds = torch.argmax(outputs, dim=1)  

            correct += (preds == labels).sum().item()
            total += labels.numel()

    print("Accuracy:", correct / total * 100, "%")

evaluate(model, loader)

Accuracy: 74.67293754851002 %
